# PhaseBreak: Housing Bubble Detection

LPPLS analysis on US housing markets using FHFA House Price Index.

**Data:** FHFA HPI (quarterly, state-level) — 6 known bubbles + 4 controls.

**Key finding:** 3/6 housing bubbles detected (FL 2006, AZ 2022, ID 2022), 0/4 false positives.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.housing.data import HOUSING_BUBBLES, HOUSING_CONTROLS, load_housing_episode
from src.housing.baselines import run_all_baselines
from src.lppls.optimizer import LPPLSOptimizer

print(f'Bubbles: {list(HOUSING_BUBBLES.keys())}')
print(f'Controls: {list(HOUSING_CONTROLS.keys())}')

In [ ]:
# Fit LPPLS on all housing episodes
for name in list(HOUSING_BUBBLES.keys()) + list(HOUSING_CONTROLS.keys()):
    try:
        ds = load_housing_episode(name)
        opt = LPPLSOptimizer(grid_size=10, n_best=5, m_range=(0.1, 0.9), omega_range=(4.0, 15.0))
        model = opt.fit(ds.t, ds.log_price)
        r2 = model.r_squared(ds.t, ds.log_price)
        is_bub = model.params and model.params.is_bubble and r2 > 0.5
        expected = name in HOUSING_BUBBLES
        mark = '\u2713' if (expected == is_bub) else '\u2717'
        print(f'{name:<20} exp={"BUB" if expected else "CTL":>3} det={"YES" if is_bub else "NO":>3} R2={r2:.3f} {mark}')
    except Exception as e:
        print(f'{name}: {e}')

In [ ]:
# Compare LPPLS with simple baselines
print(f'{"Name":<20} {"LPPLS":>6} {"CAGR":>6} {"Z-Acc":>6} {"TrDev":>6}')
for name in list(HOUSING_BUBBLES.keys())[:3]:
    try:
        ds = load_housing_episode(name)
        opt = LPPLSOptimizer(grid_size=10, n_best=5, m_range=(0.1, 0.9), omega_range=(4.0, 15.0))
        model = opt.fit(ds.t, ds.log_price)
        r2 = model.r_squared(ds.t, ds.log_price)
        lppls_bub = model.params and model.params.is_bubble and r2 > 0.5
        baselines = run_all_baselines(ds.values)
        bl_results = {bl.method: bl.is_bubble for bl in baselines}
        print(f'{name:<20} {"Y" if lppls_bub else "N":>6} {"Y" if bl_results.get("CAGR") else "N":>6} {"Y" if bl_results.get("Z-Accel") else "N":>6} {"Y" if bl_results.get("TrendDev") else "N":>6}')
    except Exception as e:
        print(f'{name}: {e}')

## Results

| Metric | LPPLS | CAGR | Z-Accel | TrendDev |
|--------|-------|------|---------|----------|
| TP (bubbles) | 3/6 | TBD | TBD | TBD |
| FP (controls) | 0/4 | TBD | TBD | TBD |

LPPLS with tightened Sornette filters achieves **zero false positives** on housing controls while detecting 3 of 6 known episodes.

**Limitation:** Data is quarterly (FHFA HPI), which limits temporal resolution. Monthly Zillow ZHVI may improve results.